# RAG-Powered Document Assistant — Core Track
### Domain: Machine Learning Study Assistant

This notebook covers **Phase 0 (environment check), Phase 1 (data collection/verification) and Phase 2 (build & evaluate the RAG pipeline)** of the graduation project, following the official project guideline (Core Track — text-only RAG, no Computer Vision/YOLO).

Source documents: 4 Machine Learning lecture PDFs placed in `data/documents/`.

**Instructions before running:**
1. Make sure Ollama is installed and running (`ollama --version`), and that you have pulled a model, e.g. `ollama pull llama3.2`.
2. Place your 4 ML lecture PDFs inside `data/documents/`.
3. Run all cells top to bottom (Kernel → Restart & Run All should work with no manual intervention).


## Phase 0 — Environment Setup (sanity check)

Verifies that the required tools/libraries from the project guideline are installed:

```
pip install jupyter pandas numpy chromadb sentence-transformers pypdf ollama python-dotenv
```


In [1]:
!pip install jupyter pandas numpy chromadb sentence-transformers pypdf ollama python-dotenv

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Abdok\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
# Phase 0 — Verify environment
import sys
import importlib

required_packages = [
    "pandas", "numpy", "chromadb", "sentence_transformers", "pypdf", "ollama", "dotenv"
]

print(f"Python version: {sys.version}\n")

missing = []
for pkg in required_packages:
    try:
        importlib.import_module(pkg)
        print(f"[OK] {pkg}")
    except ImportError:
        print(f"[MISSING] {pkg}")
        missing.append(pkg)

if missing:
    print("\nInstall missing packages with:")
    print(f"pip install {' '.join(missing)}")
else:
    print("\nAll required packages are available.")


Python version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]

[OK] pandas
[OK] numpy
[OK] chromadb
[OK] sentence_transformers
[OK] pypdf
[OK] ollama
[OK] dotenv

All required packages are available.


In [17]:
# Verify Ollama is installed and reachable
import subprocess

try:
    result = subprocess.run(["ollama", "--version"], capture_output=True, text=True, timeout=10)
    print(result.stdout.strip() or result.stderr.strip())
except FileNotFoundError:
    print("Ollama CLI not found on PATH. Install it from https://ollama.com before continuing.")
except Exception as e:
    print(f"Could not check Ollama version: {e}")


ollama version is 0.34.2


## Configuration

Central place for the settings this notebook (and later the backend) will rely on. These same values are written out in `2.7 Export` so the backend can load them without re-deriving anything.


In [3]:
from pathlib import Path

# --- Paths ---
DOCS_DIR = Path("../data/documents")
VECTOR_STORE_DIR = Path("../data/vector_store")
CONFIG_PATH = VECTOR_STORE_DIR / "config.json"

# --- Chunking ---
CHUNK_SIZE_WORDS = 500
CHUNK_OVERLAP_WORDS = 50

# --- Embeddings ---
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

# --- Vector store ---
COLLECTION_NAME = "ml_lecture_notes"

# --- LLM (Ollama) ---
OLLAMA_MODEL = "llama3.2"  # change to whichever model you have pulled locally (ollama pull <model>)

# --- Retrieval ---
TOP_K = 3

VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Documents dir: {DOCS_DIR.resolve()}")
print(f"Vector store dir: {VECTOR_STORE_DIR.resolve()}")


Documents dir: D:\RAG\data\documents
Vector store dir: D:\RAG\data\vector_store


## 2.1 Load & Inspect

Loads every PDF in `data/documents/`, extracts text page by page with `pypdf`, and records anything that fails to parse or looks like it needs OCR (no extractable text).


In [4]:
from pypdf import PdfReader

def load_pdf(path: Path):
    """Extract text per page from a single PDF. Returns (pages_text, failed, needs_ocr)."""
    pages_text = []
    failed = False
    needs_ocr = False
    try:
        reader = PdfReader(str(path))
        for page in reader.pages:
            text = page.extract_text() or ""
            pages_text.append(text)
        # Heuristic: if almost no text was extracted across all pages, it's likely a scanned PDF
        total_chars = sum(len(t.strip()) for t in pages_text)
        if total_chars < 20 * max(len(pages_text), 1):
            needs_ocr = True
    except Exception as e:
        print(f"Failed to parse {path.name}: {e}")
        failed = True
    return pages_text, failed, needs_ocr

pdf_paths = sorted(DOCS_DIR.glob("*.pdf"))
print(f"Found {len(pdf_paths)} PDF file(s) in {DOCS_DIR}:")
for p in pdf_paths:
    print(f"  - {p.name}")

documents = []  # list of dicts: {source, pages: [text, ...]}
failed_files = []
ocr_needed_files = []

for path in pdf_paths:
    pages_text, failed, needs_ocr = load_pdf(path)
    if failed:
        failed_files.append(path.name)
        continue
    if needs_ocr:
        ocr_needed_files.append(path.name)
    documents.append({"source": path.name, "pages": pages_text})

total_pages = sum(len(d["pages"]) for d in documents)
print(f"\nSuccessfully loaded: {len(documents)} document(s), {total_pages} page(s) total")
print(f"Failed to parse: {failed_files if failed_files else 'none'}")
print(f"Likely needs OCR (very little/no extractable text): {ocr_needed_files if ocr_needed_files else 'none'}")


Found 4 PDF file(s) in ..\data\documents:
  - Lecture-1-Introduction-to-ML.pdf
  - Lecture-2--Linear-Regression-.pdf
  - Lecture-3---Optimization-Algorithms.pdf
  - Lecture-4---LOGISTIC-REGREssion.pdf

Successfully loaded: 4 document(s), 238 page(s) total
Failed to parse: none
Likely needs OCR (very little/no extractable text): none


In [5]:
# Per-document inspection summary
import pandas as pd

inspection_rows = []
for d in documents:
    n_pages = len(d["pages"])
    n_chars = sum(len(p) for p in d["pages"])
    inspection_rows.append({
        "document": d["source"],
        "format": "PDF",
        "pages": n_pages,
        "extracted_characters": n_chars,
    })

inspection_df = pd.DataFrame(inspection_rows)
inspection_df


,document,format,pages,extracted_characters
0,Lecture-1-Introduction-to-ML.pdf,PDF,52,24838
1,Lecture-2--Linear-Regression-.pdf,PDF,66,22981
2,Lecture-3---Optimization-Algorithms.pdf,PDF,53,29876
3,Lecture-4---LOGISTIC-REGREssion.pdf,PDF,67,28393


### Dataset Inspection (markdown summary)

> Fill this in based on the output above once your 4 lecture PDFs are in `data/documents/`. Example of what this should read like:
>
> "The dataset contains **4 PDF documents** (ML lecture notes) with **NN pages** in total. All documents are PDF format and their text was successfully extracted using `pypdf`. Files that failed to parse: *none / list them*. Files that appear to need OCR (little/no extractable text): *none / list them*."

Run the cell below to auto-generate this summary from the actual data instead of writing it by hand.


In [6]:
summary_text = (
    f"The dataset contains {len(documents)} PDF document(s) "
    f"({', '.join(d['source'] for d in documents) if documents else 'none loaded yet'}) "
    f"with {total_pages} page(s) in total. All documents are PDF format. "
    f"Text extraction was performed using pypdf. "
    f"Files that failed to parse: {', '.join(failed_files) if failed_files else 'none'}. "
    f"Files that appear to need OCR (little/no extractable text): {', '.join(ocr_needed_files) if ocr_needed_files else 'none'}."
)
print(summary_text)


The dataset contains 4 PDF document(s) (Lecture-1-Introduction-to-ML.pdf, Lecture-2--Linear-Regression-.pdf, Lecture-3---Optimization-Algorithms.pdf, Lecture-4---LOGISTIC-REGREssion.pdf) with 238 page(s) in total. All documents are PDF format. Text extraction was performed using pypdf. Files that failed to parse: none. Files that appear to need OCR (little/no extractable text): none.


## 2.2 Chunking Strategy

**Strategy chosen: fixed-size chunking with overlap, at the word level.**

- **Chunk size:** 500 words
- **Overlap:** 50 words

**Justification:** ML lecture notes mix short definitions with multi-paragraph explanations (e.g. deriving a cost function), so a fixed word count is a simple, predictable way to keep each chunk small enough for accurate retrieval and small enough to fit comfortably in the LLM's context window alongside several other chunks, while still holding a complete idea most of the time. The 50-word overlap (10% of the chunk size) reduces the chance that a definition or explanation that spans a chunk boundary gets cut in half and becomes unretrievable as a coherent unit — a concept introduced near the end of one chunk still appears at the start of the next.


In [7]:
def chunk_text(text: str, chunk_size: int, overlap: int):
    words = text.split()
    if not words:
        return []
    chunks = []
    step = max(chunk_size - overlap, 1)
    for start in range(0, len(words), step):
        chunk_words = words[start:start + chunk_size]
        if not chunk_words:
            break
        chunks.append(" ".join(chunk_words))
        if start + chunk_size >= len(words):
            break
    return chunks

# Build chunks for every document, keeping track of source + page for citation-style grounding
all_chunks = []  # list of dicts: {id, text, source, page}
chunk_counter = 0

for d in documents:
    for page_num, page_text in enumerate(d["pages"], start=1):
        page_chunks = chunk_text(page_text, CHUNK_SIZE_WORDS, CHUNK_OVERLAP_WORDS)
        for c in page_chunks:
            all_chunks.append({
                "id": f"chunk_{chunk_counter}",
                "text": c,
                "source": d["source"],
                "page": page_num,
            })
            chunk_counter += 1

print(f"Total chunks created: {len(all_chunks)}")
if all_chunks:
    print("\nExample chunk:")
    print(f"  id: {all_chunks[0]['id']}")
    print(f"  source: {all_chunks[0]['source']} (page {all_chunks[0]['page']})")
    print(f"  text (first 200 chars): {all_chunks[0]['text'][:200]}...")


Total chunks created: 238

Example chunk:
  id: chunk_0
  source: Lecture-1-Introduction-to-ML.pdf (page 1)
  text (first 200 chars): Dr. Ahmed Hesham Mostafa Faculty of Computers and AI Helwan University Lecture 1 Machine Learning...


## 2.3 Embeddings & Vector Store

Generates an embedding for every chunk with `sentence-transformers` and stores them in a **Chroma** persistent collection on disk (`data/vector_store/`), so the FastAPI backend (built in a later phase) can load it directly without rebuilding anything at request time.


In [8]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"Loaded embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Abdok\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Abdok\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded embedding model: all-MiniLM-L6-v2
Embedding dimension: 384


C:\Users\Abdok\AppData\Local\Temp\ipykernel_12360\871244620.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")


In [9]:
import chromadb

chroma_client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))

# Start fresh each run so re-running the notebook top-to-bottom doesn't duplicate chunks
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.create_collection(name=COLLECTION_NAME)
print(f"Created Chroma collection '{COLLECTION_NAME}' at {VECTOR_STORE_DIR.resolve()}")


Created Chroma collection 'ml_lecture_notes' at D:\RAG\data\vector_store


In [10]:
# Embed all chunks and add them to the vector store (in batches)
BATCH_SIZE = 64

if not all_chunks:
    print("No chunks to embed — add PDFs to data/documents/ and re-run from 2.1.")
else:
    for start in range(0, len(all_chunks), BATCH_SIZE):
        batch = all_chunks[start:start + BATCH_SIZE]
        texts = [c["text"] for c in batch]
        embeddings = embedding_model.encode(texts, show_progress_bar=False).tolist()
        collection.add(
            ids=[c["id"] for c in batch],
            embeddings=embeddings,
            documents=texts,
            metadatas=[{"source": c["source"], "page": c["page"]} for c in batch],
        )
    print(f"Added {len(all_chunks)} chunks to the vector store.")
    print(f"Collection count: {collection.count()}")


Added 238 chunks to the vector store.
Collection count: 238


## 2.4 Retrieval & Prompting

Implements the retrieval function, builds the prompt template that combines retrieved context with the user's question, and adds citation-style grounding (the answer references which document/chunk it came from). The prompt explicitly instructs the model to only answer from the provided context, and to say so when the answer isn't in the documents — this is the core anti-hallucination requirement from the guideline.


In [11]:
def retrieve(question: str, top_k: int = TOP_K):
    query_embedding = embedding_model.encode([question]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=top_k)

    retrieved = []
    ids = results.get("ids", [[]])[0]
    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    dists = results.get("distances", [[]])[0] if results.get("distances") else [None] * len(ids)

    for cid, text, meta, dist in zip(ids, docs, metas, dists):
        retrieved.append({
            "id": cid,
            "text": text,
            "source": meta.get("source"),
            "page": meta.get("page"),
            "distance": dist,
        })
    return retrieved


In [12]:
def build_prompt(question: str, retrieved_chunks: list):
    context_blocks = []
    for i, chunk in enumerate(retrieved_chunks, start=1):
        context_blocks.append(
            f"[{i}] (source: {chunk['source']}, page {chunk['page']})\n{chunk['text']}"
        )
    context_text = "\n\n".join(context_blocks) if context_blocks else "No context retrieved."

    prompt = f"""You are a helpful Machine Learning study assistant. Answer the question using ONLY the context provided below.

Rules:
- Base your answer strictly on the provided context.
- If the answer cannot be found in the context, say clearly that the information is not available in the documents. Do not make anything up.
- After the answer, list the sources (document name and page) you used, in the format: Sources: <source> (page <page>), ...

Context:
{context_text}

Question:
{question}

Answer:"""
    return prompt


In [18]:
import ollama

def generate_answer(question: str, top_k: int = TOP_K):
    retrieved_chunks = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, retrieved_chunks)

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    answer_text = response["message"]["content"]

    sources = [f"{c['source']} (page {c['page']})" for c in retrieved_chunks]
    return {
        "question": question,
        "answer": answer_text,
        "sources": sources,
        "retrieved_chunks": retrieved_chunks,
    }


### Test retrieval + generation against 10 sample questions

Edit the list below to match the actual content of your 4 ML lecture PDFs before treating the output as final.


In [ ]:
sample_questions = [
    "What is linear regression used for?",
    "What is the difference between supervised and unsupervised learning?",
    "How does logistic regression differ from linear regression?",
    "What is overfitting and how can it be avoided?",
    "What is a decision tree?",
    "What is the purpose of a loss function / cost function?",
    "What is gradient descent?",
    "What is the difference between classification and regression?",
    "What is cross-validation used for?",
    "What is a confusion matrix?",
]

print(f"Number of test questions: {len(sample_questions)}")

test_results = []
for q in sample_questions:
    result = generate_answer(q)
    test_results.append(result)
    print(f"\nQ: {q}")
    print(f"A: {result['answer']}")
    print(f"Sources: {result['sources']}")
    print("-" * 80)


Number of test questions: 10

Q: What is linear regression used for?
A: Linear regression is used for predictive analysis and to model the relationship between a dependent variable and an independent variable, helping to predict outcomes by fitting a straight line to observed data points.

Sources: Lecture-2--Linear-Regression-.pdf (page 2), Lecture-2--Linear-Regression-.pdf (page 4), Lecture-2--Linear-Regression-.pdf (page 5)
Sources: ['Lecture-2--Linear-Regression-.pdf (page 3)', 'Lecture-2--Linear-Regression-.pdf (page 4)', 'Lecture-2--Linear-Regression-.pdf (page 5)']
--------------------------------------------------------------------------------

Q: What is the difference between supervised and unsupervised learning?
A: The difference between supervised and unsupervised learning is as follows:

Supervised learning is a type of machine learning where an algorithm learns to map input data (X) to known output labels (y). It's called "supervised" because the process of an algorithm l

## 2.6 Evaluation

Builds a results table for the same 10 test questions (question / retrieved source / answer / correct or not), then summarizes the main failure cases observed and how they were mitigated.

**Correctness is graded manually** — set `manual_correct` for each question below after reading the generated answer against your actual lecture PDFs.


In [20]:
# Fill in True/False for each question after reading the answers generated above.
# Order must match `sample_questions` / `test_results`.
manual_correct = [None] * len(test_results)  # e.g. [True, True, False, True, ...]

eval_rows = []
for result, correct in zip(test_results, manual_correct):
    top_source = result["sources"][0] if result["sources"] else "N/A"
    eval_rows.append({
        "question": result["question"],
        "retrieved_source": top_source,
        "answer": (result["answer"][:150] + "...") if len(result["answer"]) > 150 else result["answer"],
        "correct": correct,
    })

evaluation_df = pd.DataFrame(eval_rows)
evaluation_df


,question,retrieved_source,answer,correct
0,What is linear regression used for?,Lecture-2--Linear-Regression-.pdf (page 3),Linear regression is used for predictive analy...,None
1,What is the difference between supervised and ...,Lecture-1-Introduction-to-ML.pdf (page 30),The difference between supervised and unsuperv...,None
2,How does logistic regression differ from linea...,Lecture-4---LOGISTIC-REGREssion.pdf (page 32),Logistic regression differs from linear regres...,None
3,What is overfitting and how can it be avoided?,Lecture-1-Introduction-to-ML.pdf (page 29),Overfitting occurs when a model learns the tra...,None
4,What is a decision tree?,Lecture-1-Introduction-to-ML.pdf (page 23),A decision tree is a type of model used for co...,None
5,What is the purpose of a loss function / cost ...,Lecture-2--Linear-Regression-.pdf (page 41),The purpose of a loss function / cost function...,None
6,What is gradient descent?,Lecture-2--Linear-Regression-.pdf (page 46),Gradient Descent is a first-order iterative op...,None
7,What is the difference between classification ...,Lecture-4---LOGISTIC-REGREssion.pdf (page 4),Classification and regression are two types of...,None
8,What is cross-validation used for?,Lecture-1-Introduction-to-ML.pdf (page 16),Cross-validation is used to determine the effi...,None
9,What is a confusion matrix?,Lecture-1-Introduction-to-ML.pdf (page 20),A confusion matrix is not explicitly mentioned...,None


In [21]:
# Save the full evaluation table (untruncated answers) alongside the notebook for the README later
evaluation_full_df = pd.DataFrame([
    {
        "question": r["question"],
        "retrieved_source": r["sources"][0] if r["sources"] else "N/A",
        "answer": r["answer"],
        "correct": c,
    }
    for r, c in zip(test_results, manual_correct)
])
eval_csv_path = VECTOR_STORE_DIR.parent / "evaluation_results.csv"
evaluation_full_df.to_csv(eval_csv_path, index=False)
print(f"Saved evaluation table to {eval_csv_path.resolve()}")


Saved evaluation table to D:\RAG\data\evaluation_results.csv


### Failure cases observed and mitigation

> Fill in after reviewing the 10 answers above against your actual lecture content. Example structure to follow:
>
> - **Failure case 1:** *(e.g. a question whose answer wasn't in any of the 4 lecture PDFs)* — the model correctly stated the information wasn't available in the documents, thanks to the explicit "if the answer cannot be found in the context, say so" instruction in the prompt.
> - **Failure case 2:** *(e.g. retrieval pulled a chunk from the wrong lecture because two lectures use similar terminology)* — mitigated by lowering `CHUNK_SIZE_WORDS` / increasing `TOP_K` so more context is available, or by making chunk boundaries respect section headers instead of a fixed word count.
> - **Failure case 3:** *(e.g. the answer was grounded but too verbose/copied large blocks of text)* — mitigated by tightening the prompt instructions.


## 2.7 Export

The Chroma vector store is already persisted to disk at `data/vector_store/` (written incrementally as chunks were added in 2.3 — Chroma's `PersistentClient` writes through automatically). This step additionally saves the pipeline configuration (chunk size, overlap, embedding model name, collection name, top-k) as `config.json` next to it, so the backend can load everything without rebuilding or re-deriving any of it.


In [22]:
import json

config = {
    "embedding_model_name": EMBEDDING_MODEL_NAME,
    "chunk_size_words": CHUNK_SIZE_WORDS,
    "chunk_overlap_words": CHUNK_OVERLAP_WORDS,
    "collection_name": COLLECTION_NAME,
    "top_k": TOP_K,
    "ollama_model": OLLAMA_MODEL,
    "num_documents": len(documents),
    "num_chunks": len(all_chunks),
}

with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)

print(f"Saved config to {CONFIG_PATH.resolve()}:")
print(json.dumps(config, indent=2))
print(f"\nPersisted vector store directory: {VECTOR_STORE_DIR.resolve()}")
print("The backend (Phase 3) should point VECTOR_STORE_PATH at this folder and load config.json at startup.")


Saved config to D:\RAG\data\vector_store\config.json:
{
  "embedding_model_name": "all-MiniLM-L6-v2",
  "chunk_size_words": 500,
  "chunk_overlap_words": 50,
  "collection_name": "ml_lecture_notes",
  "top_k": 3,
  "ollama_model": "llama3.2",
  "num_documents": 4,
  "num_chunks": 238
}

Persisted vector store directory: D:\RAG\data\vector_store
The backend (Phase 3) should point VECTOR_STORE_PATH at this folder and load config.json at startup.
